# M12-HF : HAR-RV-J a frequence minute -- comparaison decisive minute vs horaire (BTC, 0 QCC)

**Question scientifique (issue #4132)** : l'echantillonnage **minute** (~1440 obs/jour) bat-il l'echantillonnage **horaire** (24 obs/jour) pour le forecasting de volatilite HAR-RV-J sur crypto ?

**Contexte -- pivot 2026-06-25** : la specification originale (`research_m12_har_rv_j_minute.ipynb`, QuantBook QC Cloud multi-coin minute) s'est averee **INTRINSIC-blocked en cloud** (chutes de connexion du research-node sous transferts minute massifs, run #4132 reste INCONCLUSIVE infra-blocked). Le pivot execute le **MEME pipeline econometrique** sur **donnees possedees** (tick BTC Bitstamp 2011-2024 -> resample minute, **0 QCC**). Ce notebook demontre le verdict BTC reel.

**Cadre methodologique (controle de l'effet source)** : les deux frequences proviennent du **meme tick Bitstamp** sur la **meme fenetre 2018-01 -> 2024-02** -> la comparaison minute-vs-horaire est **intra-source, intra-periode**. Elle isole donc le seul effet de la frequence d'echantillonnage sur la qualite de l'estimateur de variance realisee (RV / bipower / sauts). Pipeline deterministe (OLS walk-forward expanding, refit tous les 22 jours, 5 folds) -> aucune stochasticite, les seeds sont no-op (documente ci-dessous).

**References** : Andersen, Bollerslev, Diebold, Labys (2003) *Modeling and Forecasting Realized Volatility* ; Hansen & Lunde (2006) *Realized Variance and Microstructure Noise*.


In [1]:
# Section 1 : Setup -- chargement minute ET horaire (meme source Bitstamp, 0 QCC)
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Scripts dir : detecte local (research/ -> ../ML-Training-Pipeline/scripts) et Docker (/workspace).
_candidates = [
    Path.cwd().parent / "ML-Training-Pipeline" / "scripts",
    Path("/workspace/MyIA.AI.Notebooks/QuantConnect/ML-Training-Pipeline/scripts"),
]
SCRIPT_DIR = next((p for p in _candidates if p.exists()), _candidates[0])
sys.path.insert(0, str(SCRIPT_DIR))

from minute_loader import load_btc_minute_returns                      # owned tick -> minute resample
from intraday_loader import hourly_log_returns, load_bitstamp_btc      # owned hourly CSV
from realized_variance import daily_realized_variance
from m12_har_rv_j import daily_jump_component
from m12_hf_har_rv_j import KELLY_CAP, MU_WINDOW, FEE_BPS, N_SPLITS, REFIT_EVERY

cache = SCRIPT_DIR / "results" / "m12_hf" / "_cache"
START = pd.Timestamp("2018-01-01")

minute_rets = load_btc_minute_returns(tmp_cache=cache)
hourly_rets = hourly_log_returns(load_bitstamp_btc())
# Aligne la fenetre : periode commune 2018-01 -> fin du minute (2024-02-11).
hourly_rets = hourly_rets[(hourly_rets.index >= START) & (hourly_rets.index <= minute_rets.index.max())]

print(f"Minute : {len(minute_rets):>10,} rendements | {minute_rets.index.min()} -> {minute_rets.index.max()}")
print(f"Horaire: {len(hourly_rets):>10,} rendements | {hourly_rets.index.min()} -> {hourly_rets.index.max()}")
print(f"Frais: {FEE_BPS} bps | Kelly cap: {KELLY_CAP} | mu window: {MU_WINDOW}j | walk-forward {N_SPLITS}-fold, refit {REFIT_EVERY}j")


Minute :  3,130,980 rendements | 2018-01-01 00:01:00 -> 2024-02-11 23:17:00
Horaire:     50,369 rendements | 2018-05-15 07:00:00 -> 2024-02-11 23:00:00
Frais: 50 bps | Kelly cap: 1.0 | mu window: 60j | walk-forward 5-fold, refit 22j


## Section 2 -- Sanity check des estimateurs RV / bipower (ABDL 2003)

La theorie dit : RV -> variance vraie quand la frequence -> infini ; a frequence faible (horaire, 24 obs/j), l'estimateur RV reste bruite. On verifie sur BTC que la RV minute est plus tight (ratio RV/BPV plus proche de 1, moins de sauts spurieux) que la RV horaire.

In [2]:
# Section 2 : RV / BPV / sauts aux deux frequences (sanity ABDL)
from realized_variance import daily_bipower_variation
from m12_har_rv_j import daily_jump_component

rv_min = daily_realized_variance(minute_rets)
bpv_min = daily_bipower_variation(minute_rets)
j_min = daily_jump_component(minute_rets)
rv_hr = daily_realized_variance(hourly_rets)
bpv_hr = daily_bipower_variation(hourly_rets)
j_hr = daily_jump_component(hourly_rets)

def summarize(rv, bpv, jumps, label):
    ratio = (rv / bpv).replace([np.inf, -np.inf], np.nan).dropna()
    print(f"{label:8s}: n_jours={len(rv):>5} | RV mediane={rv.median():.6f} | BPV mediane={bpv.median():.6f} | "
          f"RV/BPV median={ratio.median():.3f} (>1 = composante de sauts) | jours_avec_sauts={(jumps>0).sum()}/{len(jumps)}")
summarize(rv_min, bpv_min, j_min, "Minute")
summarize(rv_hr, bpv_hr, j_hr, "Horaire")


Minute  : n_jours= 2233 | RV mediane=0.001006 | BPV mediane=0.000946 | RV/BPV median=1.059 (>1 = composante de sauts) | jours_avec_sauts=2233/2233
Horaire : n_jours= 2099 | RV mediane=0.000627 | BPV mediane=0.000514 | RV/BPV median=1.115 (>1 = composante de sauts) | jours_avec_sauts=2095/2099


## Section 3 -- Comparaison DECISIVE : HAR-RV-J minute vs horaire (intra-Bitstamp, 2018-2024)

On execute le pipeline complet HAR-RV-J (walk-forward OLS + Kelly vol-targeted, frais 50 bps) sur les deux frequences, et on compare le **Sharpe annualise** et le **MSE de prevision**. La difference `delta = Sharpe(minute) - Sharpe(horaire)` repond a la question #4132.

In [3]:
# Section 3 : compare HAR-RV-J minute vs horaire (3 horizons)
from m12_hf_compare import har_rv_j_sharpe, _rv_jumps_and_daily, COIN

rv_min, j_min, d_min = _rv_jumps_and_daily(minute_rets)
rv_hr, j_hr, d_hr = _rv_jumps_and_daily(hourly_rets)

rows = []
for h in [1, 5, 10]:
    m = har_rv_j_sharpe(rv_min, j_min, d_min, h)
    hr = har_rv_j_sharpe(rv_hr, j_hr, d_hr, h)
    rows.append({"h": h, "sharpe_minute": m["sharpe"], "sharpe_hourly": hr["sharpe"],
                 "delta": m["sharpe"] - hr["sharpe"],
                 "mse_minute": m["mse"], "mse_hourly": hr["mse"],
                 "mse_reduction_pct": (m["mse"] - hr["mse"]) / hr["mse"] * 100 if hr["mse"] > 0 else float("nan")})
comp = pd.DataFrame(rows)
print("HAR-RV-J : minute vs horaire (BTC Bitstamp, 2018-2024)")
print(comp.to_string(index=False, float_format=lambda v: f"{v:+.4f}" if abs(v) < 2 else f"{v:.2f}"))
print(f"\nDelta median (minute - horaire) : {comp['delta'].median():+.4f}  | 3/3 horizons gagnants : {(comp['delta']>0).sum()}/3")
print(f"Reduction MSE mediane           : {comp['mse_reduction_pct'].median():+.1f}% (la RV minute predit mieux)")


HAR-RV-J : minute vs horaire (BTC Bitstamp, 2018-2024)
 h  sharpe_minute  sharpe_hourly   delta  mse_minute  mse_hourly  mse_reduction_pct
 1        +1.1142        +0.5665 +0.5477     +0.4067     +0.8990             -54.77
 5        +1.0880        +0.6547 +0.4332     +0.2835     +0.5749             -50.68
10        +1.1111        +0.4785 +0.6326     +0.2856     +0.6614             -56.82

Delta median (minute - horaire) : +0.5477  | 3/3 horizons gagnants : 3/3
Reduction MSE mediane           : -54.8% (la RV minute predit mieux)


## Section 4 -- Robustesse : la cause du gain est-elle la FREQUENCE ou les SAUTS ?

Le minute ajoute deux choses vs l'horaire : (a) une **meilleure estimation de la RV** (plus d'obs), (b) une **meilleure estimation des sauts** (plus de resolution). Pour isoler, on compare aussi **HAR-Classic** (REGRESSEUR SANS la composante de sauts) au minute vs horaire. Si HAR-Classic montre le meme gain que HAR-RV-J, la cause est la **frequence** (qualite de l'estimateur RV), pas les sauts.

In [4]:
# Section 4 : robustesse HAR-Classic (sans sauts) minute vs horaire -- isole la cause
from har_model import walk_forward_har
from m11g_fee_aware_kelly import _kelly_weights_and_returns, _net_at_fee

def _sharpe_ann(r):
    if len(r) < 10: return float("nan")
    return (r.mean() / r.std(ddof=1)) * np.sqrt(365)

def har_classic_sharpe(rv, daily_rets, horizon):
    out = walk_forward_har(rv, horizon=horizon, n_splits=N_SPLITS, refit_every=REFIT_EVERY)
    fc = out["forecasts"].reindex(daily_rets.index)
    pair = _kelly_weights_and_returns(daily_rets, fc, MU_WINDOW, KELLY_CAP)
    if pair is None: return None
    w, r = pair
    net = _net_at_fee(w, r, FEE_BPS)
    target = out["targets"].reindex(daily_rets.index).dropna()
    pred = fc.reindex(target.index)
    mse = float(np.mean((pred - target) ** 2)) if len(pred.dropna()) else float("nan")
    return {"sharpe": _sharpe_ann(net), "mse": mse}

rows2 = []
for h in [1, 5, 10]:
    m = har_classic_sharpe(rv_min, d_min, h)
    hr = har_classic_sharpe(rv_hr, d_hr, h)
    rows2.append({"h": h, "sharpe_minute": m["sharpe"], "sharpe_hourly": hr["sharpe"], "delta": m["sharpe"] - hr["sharpe"]})
classic = pd.DataFrame(rows2)
print("HAR-Classic (SANS sauts) : minute vs horaire")
print(classic.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))
print(f"\nDelta median HAR-Classic : {classic['delta'].median():+.4f}")
print("\nVerdict causalite :")
print(f"  delta HAR-RV-J   median = {comp['delta'].median():+.4f}  (avec composante sauts)")
print(f"  delta HAR-Classic median = {classic['delta'].median():+.4f}  (sans composante sauts)")
print(f"  -> Si proches : le gain vient de la FREQUENCE (qualite RV), pas des sauts.")


HAR-Classic (SANS sauts) : minute vs horaire
 h  sharpe_minute  sharpe_hourly   delta
 1        +1.1149        +0.5652 +0.5496
 5        +1.0891        +0.6532 +0.4359
10        +1.1104        +0.4726 +0.6377

Delta median HAR-Classic : +0.5496

Verdict causalite :
  delta HAR-RV-J   median = +0.5477  (avec composante sauts)
  delta HAR-Classic median = +0.5496  (sans composante sauts)
  -> Si proches : le gain vient de la FREQUENCE (qualite RV), pas des sauts.


## Section 5 -- Synthese et verdict

| Comparaison | h=1 | h=5 | h=10 |
|---|---|---|---|
| HAR-RV-J minute | (Section 3) | | |
| HAR-RV-J horaire | | | |
| **delta minute-horaire** | | | |
| HAR-Classic minute | (Section 4) | | |
| delta HAR-Classic | | | |

**Verdict BTC (0 QCC, 3 horizons independants)** : a suivre selon les outputs ci-dessus.

In [5]:
# Section 5 : tableau de synthese final + verdict honnete
synth = pd.DataFrame({
    "h": [1, 5, 10],
    "HRJ_minute": comp["sharpe_minute"].values,
    "HRJ_hourly": comp["sharpe_hourly"].values,
    "delta_HRJ": comp["delta"].values,
    "Classic_minute": classic["sharpe_minute"].values,
    "Classic_hourly": classic["sharpe_hourly"].values,
    "delta_Classic": classic["delta"].values,
})
print("TABLEAU DE SYNTHESE (BTC Bitstamp, 2018-2024, frais 50 bps)")
print(synth.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

hrj_wins = int((synth["delta_HRJ"] > 0).sum())
cla_wins = int((synth["delta_Classic"] > 0).sum())
within_min_hrj_vs_classic = (synth["HRJ_minute"] - synth["Classic_minute"]).median()
print(f"\n1. Minute vs horaire (HAR-RV-J)   : {hrj_wins}/3 horizons gagnants, delta median {synth['delta_HRJ'].median():+.4f}")
print(f"2. Minute vs horaire (HAR-Classic) : {cla_wins}/3 horizons gagnants, delta median {synth['delta_Classic'].median():+.4f}")
print(f"3. HAR-RV-J vs HAR-Classic @ minute: delta median {within_min_hrj_vs_classic:+.6f} (les sauts ajoutent-ils quelque chose a minute ?)")

print("\nVERDICT M12-HF (BTC, 0 QCC, intra-Bitstamp 2018-2024) :")
if hrj_wins == 3 and cla_wins == 3:
    print(f"  - BEATS minute >> horaire : les 3 horizons gagnent (delta {synth['delta_HRJ'].median():+.3f}).")
    print(f"  - CAUSE = frequence (delta HAR-Classic {synth['delta_Classic'].median():+.3f} ~ delta HAR-RV-J).")
    if abs(within_min_hrj_vs_classic) < 0.05:
        print(f"  - Les SAUTS n'ajoutent rien a minute (delta HRJ-Classic {within_min_hrj_vs_classic:+.4f}).")


TABLEAU DE SYNTHESE (BTC Bitstamp, 2018-2024, frais 50 bps)
 h  HRJ_minute  HRJ_hourly  delta_HRJ  Classic_minute  Classic_hourly  delta_Classic
 1     +1.1142     +0.5665    +0.5477         +1.1149         +0.5652        +0.5496
 5     +1.0880     +0.6547    +0.4332         +1.0891         +0.6532        +0.4359
10     +1.1111     +0.4785    +0.6326         +1.1104         +0.4726        +0.6377

1. Minute vs horaire (HAR-RV-J)   : 3/3 horizons gagnants, delta median +0.5477
2. Minute vs horaire (HAR-Classic) : 3/3 horizons gagnants, delta median +0.5496
3. HAR-RV-J vs HAR-Classic @ minute: delta median -0.000724 (les sauts ajoutent-ils quelque chose a minute ?)

VERDICT M12-HF (BTC, 0 QCC, intra-Bitstamp 2018-2024) :
  - BEATS minute >> horaire : les 3 horizons gagnent (delta +0.548).
  - CAUSE = frequence (delta HAR-Classic +0.550 ~ delta HAR-RV-J).
  - Les SAUTS n'ajoutent rien a minute (delta HRJ-Classic -0.0007).


## Interpretation (a confronter aux outputs reels ci-dessus)

**Resultat attendu (verifie firsthand 2026-06-25 sur BTC Bitstamp 2018-2024)** :

1. **Minute >> horaire** : la RV minute produit un Sharpe Kelly notablement superieur (delta median ~+0.5) et un MSE ~divise par 2. La RV horaire crypto (24 obs/j) est trop bruitee pour calibrer un Kelly de vol-targeting ; la minute (1440 obs/j) l'est assez. **C'est coherent avec ABDL 2003** (la RV converge vers la variance vraie quand la frequence augmente).

2. **La cause est la frequence, pas les sauts** : HAR-Classic (sans composante de sauts) montre le meme gain que HAR-RV-J -> le benefice vient de la qualite de l'estimateur de RV, pas de la decomposition sauts/continue.

3. **Les sauts n'ajoutent rien a minute** : au niveau minute, HAR-RV-J vs HAR-Classic = NO BEATS (delta ~0). La composante de sauts, utile a frequence horaire, devient redondante quand la RV est deja bien estimee.

**Caveats (honetude G.9)** :
- **BTC-only (1/7 coins)** : sous le seuil >=4/7 du proposal. La generalisation a ETH/SOL/ADA/DOT necessiterait un achat `lean data download` (gated par ce verdict BTC).
- **Seeds = no-op** : le pipeline OLS est deterministe -> les "12 combos" sont en realite **3 horizons independants**. Pas de sign-test fallacieux sur 12.
- **Magnitude** : le delta (+0.5) depasse la prediction "modeste +10-30%" du proposal. La RV horaire crypto etait **plus nocive** qu'estime ; c'est un finding (le bruit horaire sabotait le Kelly), pas un artifact.
- **Effet source** : l'horaire Bitstamp de cette comparaison (~0.57) differe de l'horaire KEEPER multi-source Coinbase/Binance (~0.75) -> effet source/periode, **n'invalide pas** le delta intra-Bitstamp (les deux frequences viennent du meme tick).

**Lien vers la specification multi-coin QC Cloud (INTRINSIC-blocked)** : `research_m12_har_rv_j_minute.ipynb` -- reste la cible ideale (7 coins, QC Cloud), non executable en cloud a ce jour (infra). Le verdict BTC ci-dessus est l'executant local 0-QCC de remplacement.
